# My listening, from the play-history marts

A look at how, when and what I listen to on Spotify, built on the dbt
marts from mccoy's hourly play-history pipeline (`analytics/`), via a
parquet snapshot so this notebook needs no database.

> TODO(Evan): one-paragraph intro: why this exists and the headline
> finding, once you've run it on your own data.

**How to run it**

1. Export the real snapshot (reads `DATABASE_URL`, read-only):
   `uv run python -m scripts.export_marts` → `analysis/data/` (gitignored).
2. Set `SOURCE = "snapshot"` in the Data cell below and run all.

As committed, `SOURCE = "fixture"`: the code runs on **synthetic** data
from `analysis/fixture.py` (made-up artists). Nothing printed in fixture
mode says anything about real listening. The data code lives in
`analysis/metrics.py` and is tested in `tests/test_analysis.py`.

In [ ]:
import sys
import tempfile
from pathlib import Path

# Run from anywhere inside the repo: find the root and make it importable.
ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd
import plotly.express as px

from analysis import fixture, metrics

pd.set_option("display.precision", 2)
px.defaults.template = "plotly_white"

## 0. Data

Five marts: `fct_plays` (one row per play), `fct_listening_sessions`
(30-minute gap rule), `mart_daily_listening`, `mart_hour_of_week` and
`mart_artist_streaks`. Times are local.

> TODO(Evan): the date range this covers and anything odd about it
> (pipeline start date, gaps from outages or the API's 50-play window).

In [ ]:
# "fixture": synthetic data from analysis/fixture.py (how this was built)
# "snapshot": the real export in analysis/data/ (scripts/export_marts.py)
SOURCE = "fixture"

if SOURCE == "fixture":
    DATA_DIR = Path(tempfile.mkdtemp(prefix="mccoy-fixture-"))
    fixture.write_fixture_snapshot(DATA_DIR)
elif SOURCE == "snapshot":
    DATA_DIR = ROOT / "analysis" / "data"
else:
    raise ValueError(f"SOURCE must be fixture or snapshot, not {SOURCE!r}")

marts = metrics.load_marts(DATA_DIR)
plays = marts["fct_plays"]
sessions = marts["fct_listening_sessions"]
daily = marts["mart_daily_listening"]
how = marts["mart_hour_of_week"]
streaks = marts["mart_artist_streaks"]

print(f"SOURCE = {SOURCE!r}, reading {DATA_DIR}")
if SOURCE == "fixture":
    print("SYNTHETIC FIXTURE DATA - not real listening. Don't narrate it.")
display(pd.Series({m: len(f) for m, f in marts.items()}, name="rows"))
display(pd.Series(metrics.summary(plays), name="summary"))

## 1. How much I listen

Minutes per day with a 7-day rolling mean, then totals per week
(Monday start) with how many days in the week had any listening.

> TODO(Evan): typical day/week, trend over the period, notable spikes
> or dry spells and what was going on.

In [ ]:
day = metrics.daily_series(daily, window=7)
fig = px.bar(
    day.reset_index(),
    x="listen_date",
    y="minutes",
    labels={"listen_date": "", "minutes": "minutes"},
    title="Minutes listened per day (line: 7-day mean)",
)
fig.add_scatter(
    x=day.index, y=day["minutes_rolling"], mode="lines", name="7-day mean"
)
fig.show()

In [ ]:
week = metrics.weekly_totals(daily)
fig = px.bar(
    week.reset_index(),
    x="week",
    y="minutes",
    hover_data=["plays", "days_listened"],
    labels={"week": "week of", "minutes": "minutes"},
    title="Minutes per week",
)
fig.show()
display(week.describe())

## 2. When I listen

Minutes by local day of week and hour of day.

> TODO(Evan): the shape of a week: commute/work/evening blocks,
> weekday vs weekend, any surprises.

In [ ]:
grid = metrics.hour_of_week_grid(how, value="minutes")
fig = px.imshow(
    grid,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "hour of day", "y": "", "color": "minutes"},
    title="Minutes by day of week and hour",
)
fig.show()

In [ ]:
by_hour = grid.sum(axis=0).rename("minutes").reset_index()
fig = px.line(
    by_hour,
    x="hour",
    y="minutes",
    markers=True,
    title="Minutes by hour of day (all days)",
)
fig.show()

## 3. Sessions

A session is a run of plays with no gap over 30 minutes
(`session_gap_minutes` in `analytics/dbt_project.yml`).

> TODO(Evan): what a typical session looks like, how often it's a
> single track vs a long block, and whether that matches how it feels.

In [ ]:
display(pd.Series(metrics.session_stats(sessions), name="sessions"))
buckets = metrics.session_length_buckets(sessions)
fig = px.bar(
    x=buckets.index,
    y=buckets.values,
    labels={"x": "session length (minutes)", "y": "sessions"},
    title=f"Session lengths (n = {len(sessions)})",
)
fig.show()

## 4. Who I listen to

Top artists by plays, how concentrated listening is on them, and how
many artists each week were heard for the first time *in this data*
(not ever: the data starts when the pipeline did).

> TODO(Evan): the core rotation, how concentrated it is, and whether
> discovery is steady or comes in bursts.

In [ ]:
top = metrics.top_artists(plays, n=15)
fig = px.bar(
    top.iloc[::-1],
    x="plays",
    y="artist",
    orientation="h",
    hover_data=["minutes", "share"],
    title="Top 15 artists by plays",
)
fig.show()
for k in (1, 5, 10):
    share = metrics.concentration(plays, k=k)
    print(f"top {k:>2} artists: {share:.0%} of {len(plays)} plays")

In [ ]:
new = metrics.new_artists_by_week(plays)
fig = px.bar(
    new.reset_index(),
    x="week",
    y=["new_artists", "artists"],
    barmode="group",
    labels={"week": "week of", "value": "artists", "variable": ""},
    title="Artists per week, and how many were new to the data",
)
fig.show()

## 5. Streaks

Runs of consecutive local days on which an artist was played.

> TODO(Evan): the longest runs and what drove them (new album, a tour,
> a mood).

In [ ]:
longest = metrics.top_streaks(streaks, n=10)
display(longest)
fig = px.timeline(
    longest.assign(streak_end=longest["streak_end"] + pd.Timedelta(days=1)),
    x_start="streak_start",
    x_end="streak_end",
    y="artist_name",
    hover_data=["streak_days"],
    title="Ten longest artist streaks",
)
fig.update_yaxes(autorange="reversed", title="")
fig.show()

## 6. Repeats

The most-played tracks and how much of all listening is replays.

> TODO(Evan): the tracks on repeat, and whether replays are a big or
> small part of the total.

In [ ]:
display(metrics.top_tracks(plays, n=15))
print(f"replay share: {metrics.replay_share(plays):.0%} of plays are of "
      "a track played more than once")

## 7. Caveats

- **Coverage.** Play history comes from Spotify's recently-played
  endpoint, polled hourly; it only returns the last 50 plays, so very
  heavy hours can drop plays, and outages leave gaps.
- **Minutes are track length.** Every mart sums `duration_ms`, so a
  partly played track counts its full length; what Spotify counts as a
  play at all is Spotify's rule, not ours.
- **Local time** is `listening_tz` in `analytics/dbt_project.yml`
  (America/Los_Angeles), not wherever I physically was.
- **Small n.** Rates and shares on a few weeks of data move a lot;
  counts are printed next to them for that reason.

> TODO(Evan): confirm or correct these against the real data, and add
> anything else a reader should know before trusting the numbers.